In [0]:
%sh
nc -zv c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com 443

Connection to c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com (52.191.218.117) 443 port [tcp/https] succeeded!


In [0]:
%sh curl -s ifconfig.me

52.249.199.78

In [0]:
from sdds.common.util import NotebookUtil
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType
from databricks.sdk.runtime import dbutils

import json
import requests
from requests.auth import HTTPBasicAuth
from datetime import datetime, timedelta

catalog_name = NotebookUtil.notebook_param("sdds_catalog")
schema_name = NotebookUtil.notebook_param("sdds_bronze_schema")
table_name = "catalog-stream-dbx-bronze"

# ES connection config
es_host = "d89a8095f4ec40d8ac0443696bbcb049.eastus.azure.elastic-cloud.com"
es_port = "443"
es_user = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-user-prodauth")
es_password_key = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-password-prodauth")

es_index = "catalog-stream-load-1"

In [0]:
es_fields = [
    "parentPartnumber",
    "partnumber",
    "parentCatentryId",
    "catentryId",
    "brand",
    "color",
    "colorFamily",
    "colorSwatch",
    "colorSeq",
    "dsgId",
    "dsgIdentifier",
    "type",
    "attributes",
    "customSkuAttributes",
    "defAttributes",
    "floatFacets",
    "numberFacets",
    "searchAttributes",
    "stringFacets",
    "assetSeoUrl",
    "catgroupSeq",
    "dsgCatgroups",
    "dsgSeoUrl",
    "ggCatgroups",
    "ggSeoUrl",
    "leafCategories",
    "parentCatgroup",
    "parentCatgroup0",
    "parentCatgroup1",
    "parentCatgroup2",
    "parentCatgroup3",
    "parentCatgroup4",
    "parentCatgroup5",
    "parentCatgroup6",
    "parentCatgroup7",
    "parentCatgroup8",
    "parentCatgroup9",
    "plCatgroups",
    "plSeoUrl",
    "primaryCategories",
    "productGroup",
    "productSearchFlag",
    "seo",
    "seoURLs",
    "buyable",
    "catalogIds",
    "dsgProductSortDate",
    "dsgPublishOverride",
    "endDate",
    "endDateTime",
    "fullImage",
    "ggProductSortDate",
    "ggPublishOverride",
    "keyword",
    "longDescription",
    "mfName",
    "name",
    "onOrder",
    "comingSoonEndDateTime",
    "plProductSortDate",
    "plPublishOverride",
    "productType",
    "published",
    "startDate",
    "startDateTime",
    "taxCode",
    "thumbnail",
    "webActiveDate",
    "ggOverrides",
    "plOverrides",
    "dsgPriceIndicators",
    "ggPriceIndicators",
    "plPriceIndicators",
    "kafkaPriceList",
    "priceList",
    "dsgQuantitySold",
    "dsgTotalPriceSold",
    "dsgOverrides",
    "ggQuantitySold",
    "ggTotalPriceSold",
    "plQuantitySold",
    "plTotalPriceSold",
    "salesData",
    "ranking",
    "caliaWebActive",
    "dsgAppWebActive",
    "dsgMobileAppWebActive",
    "dsgWebActive",
    "g3WebActive",
    "ggAppWebActive",
    "ggMobileAppWebActive",
    "ggWebActive",
    "ggAkamaiRedirect",
    "ggKeywordOverride",
    "ggUrl",
    "plWebActive",
    "stackdWebActive",
    "vrstWebActive",
    "swatchPartNumber",
    "primaryUPC",
    "auxDescription2",
]

# Schema: all fields as StringType for bronze-layer raw ingestion
schema = StructType([StructField(f, StringType(), True) for f in es_fields])

# --- Step 1: Scroll ES data and write batches to temp DBFS path ---
base_url = f"https://{es_host}:{es_port}"
auth = HTTPBasicAuth(es_user, es_password_key)
headers = {"Content-Type": "application/json"}

load_ts = datetime.now()
load_timestamp = load_ts.strftime("%Y-%m-%d %H:%M:%S")

# Volume-based landing path partitioned by extraction date (from parameter widget)
volume_base = f"/Volumes/{catalog_name}/{schema_name}/{table_name}"
dbutils.widgets.text("extraction_date", datetime.now().strftime('%Y-%m-%d'))
extraction_date = dbutils.widgets.get("extraction_date")
volume_path = f"{volume_base}/extraction_date={extraction_date}"
dbutils.fs.mkdirs(volume_path)

search_body = {
    "size": 5000,
        "query": {"match_all": {}},
    "_source": es_fields
}

response = requests.post(f"{base_url}/{es_index}/_search?scroll=5m", json=search_body, auth=auth, headers=headers)
if not response.ok:
    raise RuntimeError(f"Elasticsearch search failed: {response.status_code} {response.text}")
results = response.json()
scroll_id = results["_scroll_id"]
hits = results["hits"]["hits"]
total_hits = results["hits"]["total"]["value"]
print(f"Total matching documents: {total_hits}")

batch_num = 0
total_fetched = 0

def write_batch(hits_batch, batch_id):
    """Write a batch of hits as newline-delimited JSON to DBFS."""
    records = []
    for hit in hits_batch:
        src = hit["_source"]
        # Convert all values to strings for consistent bronze-layer ingestion
        record = {k: json.dumps(v) if isinstance(v, (list, dict)) else str(v) if v is not None else None for k, v in src.items()}
        records.append(json.dumps(record))
    lines = "\n".join(records)
    dbutils.fs.put(f"{volume_path}/batch_{batch_id:05d}.json", lines, overwrite=True)
    return len(hits_batch)

# Write initial batch
if hits:
    total_fetched += write_batch(hits, batch_num)
    batch_num += 1

# Scroll through remaining results
while len(hits) > 0:
    response = requests.post(f"{base_url}/_search/scroll", json={"scroll": "5m", "scroll_id": scroll_id}, auth=auth, headers=headers)
    response.raise_for_status()
    results = response.json()
    scroll_id = results.get("_scroll_id")
    hits = results["hits"]["hits"]
    if hits:
        total_fetched += write_batch(hits, batch_num)
        batch_num += 1
        #if batch_num == 5:
        #   break
        if total_fetched % 50000 < 5000:
            print(f"  Fetched {total_fetched} / {total_hits} documents...")

# Clear scroll context
requests.delete(f"{base_url}/_search/scroll", json={"scroll_id": scroll_id}, auth=auth, headers=headers)
print(f"Fetched {total_fetched} documents in {batch_num} batches.")

if total_fetched == 0:
    dbutils.notebook.exit("No new records. Exiting.")


Total matching documents: 4460462
Wrote 574462 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
  Fetched 50000 / 4460462 documents...
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
  Fetched 100000 / 4460462 documents...
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
  Fetched 150000 / 4460462 documents...
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 146546 bytes.
Wrote 133530 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
Wrote 109999 bytes.
  Fetched 200000 / 4460462 documents...
Wrote 1

In [0]:
# --- Step 2: Read JSON files from volume and add load timestamp ---
volume_base = f"/Volumes/{catalog_name}/{schema_name}/{table_name}"
volume_path = f"{volume_base}/extraction_date={extraction_date}"

df = (
    spark.read.schema(schema).json(volume_path)
    .withColumn("load_timestamp", lit(load_timestamp).cast("timestamp"))
    .withColumn("extraction_date", lit(extraction_date).cast("date"))
    .filter(col("type").isin("style", "sku", "bundle"))
)

record_count = df.count()
print(f"DataFrame record count: {record_count}")
df.display()

DataFrame record count: 1872578


partnumber parentPartnumber type buyable name mfName keyword longDescription thumbnail fullImage published assetSeoUrl dsgSeoUrl taxCode productType startDate endDate startDateTime endDateTime webActiveDate dsgProductSortDate productSearchFlag catalogIds catentryId parentCatentryId primaryUPC searchAttributes auxDescription2 dsgPublishOverride onOrder comingSoonEndDateTime attributes defAttributes customSkuAttributes stringFacets floatFacets numberFacets priceList seoURLs productGroup primaryCategories salesData dsgPriceIndicators leafCategories parentCatgroup dsgCatgroups ggCatgroups plCatgroups parentCatgroup0-9 catgroupSeq ranking swatchPartNumber color dsgOverrides dsgQuantitySold ggQuantitySold plQuantitySold dsgTotalPriceSold ggTotalPriceSold plTotalPriceSold load_timestamp extraction_date 27400202 26ONXWCASUCLDX4DSDODI sku 1 On Women's Cloud X 4 AD On null null 26ONXWCASUCLDX4DSDODI 26ONXWCASUCLDX4DSDODI 1 /p/on-womens-cloud-x-4-ad-26onxwcasucldx4dsdodi/26onxwcasucldx4dsdodi /p/on-womens-cloud-x-4-ad-26onxwcasucldx4dsdodi/26onxwcasucldx4dsdodi 61000 Athletic & Sneakers 2026-04-07T00:00:00.000EDT 2099-12-31T00:00:00.000EST 1775534400000 4102376400000 2026-04-06T00:00:00.000EDT 02/04/2026 0 ["10001", "12301"] 27400202 3586232 7615537611709 7615537611709 Women's Rubber Construction Imported 6.0 Pink 520-002-014-001 Low 82103 3WF10174755 Medium Athletic & Sneakers Low Cross Training Sneakers Low Women's WomensRunning-129913 NewColor Sale Salt/Lily null 1 null null [{"X_BRAND":"On"},{"PRIMARY_CATEGORY_DSG ":"WomensRunning-129913"},{"X_BazaarVoice_ratings_DSG":5.0},{"X_BazaarVoice_count_DSG":3},{"X_BazaarVoice_count_ovr_DSG":3},{"X_BazaarVoice_q_count_DSG":0},{"X_BazaarVoice_a_count_DSG":0},{"X_BazaarVoice_ratings_ovr_DSG":5.0},{"3695":"Imported"},{"5588":"Rubber Construction"},{"4298":"NewColor"},{"4495":"Y"},{"2101":"Women's"},{"6021":"Brand Excluded"},{"4285":"Cross Training"},{"5471":"Medium"},{"3961":"True"},{"4187":"Sneakers"},{"4490":"https://www.dickssportinggoods.com/protips/sports-and-activities/running/how-to-choose-running-shoes"},{"5733":"Yes"},{"5495":"Women's"},{"5076":"N"},{"4493":"Y"},{"4474":"02/04/2026"},{"5382":"Athletic & Sneakers"},{"4297":"Y"},{"4482":"Y"},{"4723":"Y"}] [{"identifier": "5470", "value": "Medium/B", "seq": 50.0}, {"identifier": "5225", "value": "6.0", "seq": 370.0}, {"identifier": "298", "value": "Salt/Lily", "seq": 0.0}] [{"storeId": 10001, "value": "Laces", "key": "5940"}, {"storeId": 10001, "value": "7615537611709", "key": "PRIMARY_UPC"}, {"storeId": 10001, "value": "Women's", "key": "2101"}, {"storeId": 10001, "value": "NOT RESTRICTED", "key": "5416"}, {"storeId": 10001, "value": "NOT RESTRICTED", "key": "5415"}, {"storeId": 10001, "value": "00", "key": "3677"}, {"storeId": 10001, "value": "Imported", "key": "3695"}, {"storeId": 10001, "value": "0", "key": "3680"}, {"storeId": 10001, "value": "Brand Excluded", "key": "6021"}, {"storeId": 10001, "value": "520-002-014-001", "key": "3318"}, {"storeId": 10001, "value": "82103", "key": "3396"}, {"storeId": 10001, "value": "3WF10174755", "key": "3290"}, {"storeId": 10001, "value": "NOT RESTRICTED", "key": "5414"}, {"storeId": 10001, "value": "Yes", "key": "5733"}, {"storeId": 15108, "value": "02/04/2026", "key": "4474"}, {"storeId": 10001, "value": "N", "key": "5076"}, {"storeId": 15108, "value": "Y", "key": "4493"}, {"storeId": 10001, "value": "Y", "key": "4482"}, {"storeId": 15108, "value": "Y", "key": "4723"}, {"storeId": 10001, "value": "Low", "key": "4861"}, {"storeId": 10001, "value": "Sneakers", "key": "4187"}, {"storeId": 11201, "value": "Y", "key": "4495"}, {"storeId": 15108, "value": "https://www.dickssportinggoods.com/protips/sports-and-activities/running/how-to-choose-running-shoes", "key": "4490"}, {"storeId": 10001, "value": "True", "key": "3961"}, {"storeId": 15108, "value": "N", "key": "4299"}, {"storeId": 10001, "value": "Do Not Hide", "key": "5255"}, {"storeId": 15108, "value": "WomensRunning-129913", "key": "PRIMARY_CATEG

In [0]:
# --- Step 3: Write extraction data to Delta table, partitioned by extraction_date (idempotent) ---
target_table = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

# Create table on first run; overwrite only this partition on subsequent runs (idempotent re-runs)
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.`{table_name}`"):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("extraction_date")
        .saveAsTable(target_table)
    )
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"extraction_date = '{extraction_date}'")
        .partitionBy("extraction_date")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Wrote {record_count} records to {target_table} (partition: extraction_date = '{extraction_date}')")

Appended 1872578 records to `dev_sdsc_db`.`sdds_bronze`.`catalog-stream-dbx-bronze` (partition: extraction_date = '2026-07-02')
